# 99 · Robustness Checks

Test sensitivity of results to: event definition choices, placebo dates, sample restrictions, and alternative windows.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel  = pd.read_parquet(DATA_DIR / "ship_month_panel.parquet")
events = pd.read_csv(DATA_DIR / "category_change_events.csv", parse_dates=["event_month"])

print(f"Panel: {panel.shape}  |  Events: {len(events)}")

In [ ]:
from linearmodels.panel import PanelOLS

def run_twfe(data, outcome="log_revenue_per_berth", treatment="cabin_category_count"):
    """Quick TWFE helper — returns (coef, ci_lo, ci_hi)."""
    d = data.copy()
    d["log_berth"] = np.log(d["berth_capacity"])
    idx = d.set_index(["ship_id", "month"])
    fe = PanelOLS(
        idx[outcome],
        idx[[treatment, "log_berth"]],
        entity_effects=True, time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)
    ci = fe.conf_int().loc[treatment]
    return fe.params[treatment], ci["lower"], ci["upper"]

beta_base, lo_base, hi_base = run_twfe(panel)
print(f"Baseline TWFE β = {beta_base:.4f}  95% CI [{lo_base:.4f}, {hi_base:.4f}]")
print(f"True β          = 0.06")

## Check 1 · Restrict to δ=1 vs δ=2 expansion events

If the effect scales linearly with Δ, β should be similar across both groups.

In [ ]:
for delta in [1, 2]:
    event_ships = events[events["delta_categories"] == delta]["ship_id"].tolist()
    # Create version of panel where only these ships are treated
    p_sub = panel.copy()
    p_sub_idx = p_sub.set_index(["ship_id", "month"])
    fe_sub = PanelOLS(
        p_sub_idx["log_revenue_per_berth"],
        p_sub_idx[["cabin_category_count", "log_berth"]] if "log_berth" in p_sub_idx.columns
        else (p_sub_idx.assign(log_berth=np.log(p_sub_idx["berth_capacity"]))[["cabin_category_count", "log_berth"]]),
        entity_effects=True, time_effects=True,
    )
    # Just compare treated vs all control within that subset
    sub = panel[panel["ship_id"].isin(event_ships + panel[~panel["had_category_change"]]["ship_id"].unique().tolist())].copy()
    b, lo, hi = run_twfe(sub)
    print(f"δ={delta} events only: β = {b:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (N treated = {len(event_ships)})")
print(f"True β per category = 0.06 (DGP is linear in categories)")

## Check 2 · Placebo event dates (shift events ±6 months)

If we move all events 6 months earlier (or later), we shouldn't find an effect in the 'pre-period' of the original event.

In [ ]:
results_placebo = []

for shift in [-6, +6]:
    events_placebo = events.copy()
    events_placebo["event_month_idx_orig"] = events_placebo["event_month_idx"]
    events_placebo["event_month_idx"] = (events_placebo["event_month_idx"] + shift).clip(1, 46)

    # Recode panel post-indicator with placebo timing
    p_placebo = panel.merge(
        events_placebo[["ship_id", "event_month_idx"]].rename(
            columns={"event_month_idx": "placebo_event_idx"}
        ),
        on="ship_id", how="left"
    )
    p_placebo["month_idx"] = p_placebo.groupby("ship_id")["month"].transform(
        lambda s: (pd.to_datetime(s).dt.to_period("M") - pd.to_datetime(s).min().to_period("M")).apply(lambda x: x.n)
    )
    p_placebo["post_placebo"] = (
        (p_placebo["had_category_change"]) &
        (p_placebo["month_idx"] >= p_placebo["placebo_event_idx"])
    ).astype(float)

    p_idx = p_placebo.set_index(["ship_id", "month"])
    p_idx["log_berth"] = np.log(p_idx["berth_capacity"])
    fe = PanelOLS(
        p_idx["log_revenue_per_berth"],
        p_idx[["post_placebo", "log_berth"]],
        entity_effects=True, time_effects=True,
    ).fit(cov_type="clustered", cluster_entity=True)
    b = fe.params["post_placebo"]
    ci = fe.conf_int().loc["post_placebo"]
    results_placebo.append((shift, b, ci["lower"], ci["upper"]))
    print(f"Placebo shift {shift:+d} months: β = {b:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")

print(f"\nBaseline (correct timing):  β = {beta_base:.4f}")
print("Placebo estimates near zero or outside CI of baseline → timing-specific effect (good)")

## Check 3 · By brand tier

In [ ]:
print(f"{'Brand Tier':<12}  {'β':>8}  {'CI Low':>8}  {'CI High':>8}  {'N ships':>8}")
print("-" * 55)
for tier in ["economy", "premium", "luxury"]:
    sub = panel[panel["brand_tier"] == tier].copy()
    b, lo, hi = run_twfe(sub)
    n = sub["ship_id"].nunique()
    print(f"{tier:<12}  {b:>8.4f}  {lo:>8.4f}  {hi:>8.4f}  {n:>8}")
print(f"{'All tiers':<12}  {beta_base:>8.4f}  {lo_base:>8.4f}  {hi_base:>8.4f}  {panel['ship_id'].nunique():>8}")
print()
print("True β = 0.06 for all tiers (DGP is homogeneous — heterogeneity = estimation noise)")

## Check 4 · Leave-one-region-out

In [ ]:
regions = panel["sailing_region"].unique()
print(f"{'Excluded Region':<25}  {'β':>8}  {'CI':>20}")
print("-" * 57)
for region in regions:
    sub = panel[panel["sailing_region"] != region].copy()
    b, lo, hi = run_twfe(sub)
    print(f"{region:<25}  {b:>8.4f}  [{lo:.4f}, {hi:.4f}]")
print(f"{'None (baseline)':<25}  {beta_base:>8.4f}  [{lo_base:.4f}, {hi_base:.4f}]")

## Summary

A well-specified design should show:
- **Stable β across specifications** (similar across brand tiers, regions)
- **Placebo effects near zero** (effect is timing-specific, not spurious)
- **δ=1 and δ=2 events give similar β** (linear dose-response as expected from DGP)

In the simulated data, all checks should pass by construction — true β = 0.06 uniformly. On real data, deviations are informative.